## Loading The Data
We start by loading the 20 newsgroups dataset's raw text data, and split it into train and test.

In [2]:
from sklearn.datasets import fetch_20newsgroups

# Load the full training and testing dataset.
data_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
data_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

# Get the raw documents and labels.
X_train, y_train = data_train.data, data_train.target
X_test, y_test = data_test.data, data_test.target

# Target names (class labels):
target_names = data_train.target_names

print('Training data samples:', len(X_train))
print('Testing data samples:', len(X_test))
print('Number of classes:', len(target_names))

Training data samples: 11314
Testing data samples: 7532
Number of classes: 20


## Vectorizing The Data
We will vectorize the data in two ways, 1. TF-IDF 2.Count Vectorizer
Count Vectorizer use raw counts of words and TF-IDF gives a score based on importance, let's see the difference with a random document example.

In [32]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import numpy as np
import pandas as pd

# --------------------
# 1. Fit Vectorizers
# --------------------
count_vect = CountVectorizer()
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)


# --------------------
# 2. Pick a Single Document
# --------------------
doc_index = 5 # can choose any document index from training set
doc_text = X_train[doc_index]

# --------------------
# 3. Extract the Vector for That Document
# --------------------
doc_count_vec = X_train_count[doc_index]
doc_tfidf_vec = X_train_tfidf[doc_index]

doc_count_arr = doc_count_vec.toarray().flatten() # Convert to dense array and flatten
doc_tfidf_arr = doc_tfidf_vec.toarray().flatten() # Convert to dense array and flatten

# Get feature names
count_features = count_vect.get_feature_names_out()
tfidf_features = tfidf_vect.get_feature_names_out()

# --------------------
# 4. Find Nonzero Elements
# --------------------
nonzero_count_idx = doc_count_arr.nonzero()[0].astype(int)    # indices of nonzero tokens (Count)
nonzero_tfidf_idx = doc_tfidf_arr.nonzero()[0]    # indices of nonzero tokens (TF-IDF)


# --------------------
# 5. Combine into a Single DataFrame
# --------------------

# For a fair comparison, let's focus on words that appear at least once in this document,
# which means the union of nonzero_count_idx and nonzero_tfidf_idx.
nonzero_union = np.union1d(nonzero_count_idx, nonzero_tfidf_idx)

features = [count_features[i] for i in nonzero_union]
counts = doc_count_arr[nonzero_union]
tfidfs = doc_tfidf_arr[nonzero_union]

# 'Count' - Number of times the token(word) appears in the document
# 'TF-IDF' - TF-IDF score of the token in the document
df = pd.DataFrame({
    'token index': nonzero_union,
    'token': features,
    'count': counts,
    'tfidf': tfidfs
})

# Sort by count or tfidf (descending)
df_sorted = df.sort_values(by='count', ascending=False)
df_sorted = df_sorted.set_index('token').drop(columns=['token index']).T
df_sorted.loc['count'] = df_sorted.loc['count'].astype(int)  # make sure it's int
df_sorted.loc['tfidf'] = df_sorted.loc['tfidf'].astype(float) # ensure float

df_styled = (
    df_sorted.style
    .format("{:.0f}", subset=pd.IndexSlice[['count'], :])   # no decimals for 'count' row
    .format("{:.5f}", subset=pd.IndexSlice[['tfidf'], :])   # 4 decimals for 'tfidf' row
)

# --------------------
# 6. Display the Comparison
# --------------------
print("\n=== TOKEN COUNTS vs. TF-IDF (Single Document) ===\n")
print("Sorted by count:")
display(df_styled)



df_sorted2 = df.sort_values(by='tfidf', ascending=False)
df_sorted2 = df_sorted2.set_index('token').drop(columns=['token index']).T
df_sorted2.loc['count'] = df_sorted2.loc['count'].astype(int)  # make sure it's int
df_sorted2.loc['tfidf'] = df_sorted2.loc['tfidf'].astype(float) # ensure float

df_styled = (
    df_sorted2.style
    .format("{:.0f}", subset=pd.IndexSlice[['count'], :])   # no decimals for 'count' row
    .format("{:.5f}", subset=pd.IndexSlice[['tfidf'], :])   # 4 decimals for 'tfidf' row
)
print("Sorted by tf-idf:")
display(df_styled)



=== TOKEN COUNTS vs. TF-IDF (Single Document) ===

Sorted by count:


token,the,of,weapons,to,destruction,and,mass,in,com,you,stratus,article,foxvog,for,be,that,on,cdt,this,vtt,fi,writes,believe,it,can,from,by,douglas,rutledge,right,says,sw,ulowell,then,any,term,when,was,vttoulu,keep,us,her,first,if,would,individuals,transfer,john,many,lawrence,she,tko,means,modern,my,needless,tavares,or,should,people,re,edu,makes,doug,an,dfo,as,speak,back,shotguns,automatic,show,sign,sks,special,second,stating,street,subject,support,argument,sweeper,semi,bear,are,say,putting,quote,blank,read,reasonable,reduced,restrictions,result,rewording,rifles,biological,rigidly,rocket,bill,sarah,switching,thanks,another,usage,using,vos,about,58,4t,we,4j3,what,1r1eu1,where,with,1qv87v,write,1993apr20,year,uses,access,presenting,up,analysis,them,amendment,there,these,allowed,thousands,allegedly,all,today,topics,agree,accidental,understanding,understood,property,power,each,ideas,hard,has,have,he,defined,his,hope,deaths,point,immediately,cs,individual,crimial,investors,course,count,hands,handguns,great,government,easily,doubt,even,every,evidently,existant,f8f,don,find,does,follows,disagree,destructive,gas,given,jrutledg,cost,keeping,non,nuclear,nukes,number,cbw,cannot,only,c5n3gi,organization,other,oulu,ousrvr,out,own,packet,brady,not,nerve,killed,neighbor,later,control,lines,16899,consider,company,massive,mean,commonly,millions,coming,must,class,need,checks,083057
count,17,17,13,10,7,7,7,7,6,6,6,5,5,5,5,5,4,4,4,4,4,4,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
tfidf,0.12030,0.12769,0.42512,0.07338,0.29575,0.05490,0.23218,0.05431,0.07385,0.06139,0.22956,0.05837,0.28272,0.04347,0.05168,0.04453,0.03950,0.16189,0.03994,0.22209,0.13497,0.04438,0.06343,0.02642,0.03557,0.01986,0.03909,0.10584,0.15281,0.05809,0.07089,0.11416,0.13298,0.05141,0.03985,0.09510,0.04568,0.02511,0.11309,0.04828,0.03838,0.05635,0.03754,0.02199,0.02597,0.06784,0.06836,0.04779,0.03779,0.07227,0.05681,0.11309,0.05035,0.06546,0.02555,0.08865,0.08328,0.02204,0.03530,0.03298,0.01817,0.01887,0.05023,0.06741,0.02495,0.11309,0.02367,0.03063,0.02042,0.04825,0.03660,0.02660,0.03498,0.05552,0.02982,0.02415,0.04188,0.03279,0.00662,0.02466,0.02988,0.05464,0.03754,0.03489,0.01088,0.01913,0.03292,0.03343,0.04225,0.02169,0.03122,0.03915,0.03932,0.02974,0.05464,0.04415,0.04546,0.06113,0.03589,0.02712,0.04737,0.04176,0.01904,0.02110,0.04087,0.02109,0.04382,0.01314,0.03477,0.04659,0.01546,0.06113,0.01273,0.06382,0.01866,0.01043,0.06113,0.02786,0.03252,0.02212,0.02871,0.02443,0.04567,0.01533,0.03498,0.01644,0.03551,0.01269,0.01756,0.03188,0.03498,0.04765,0.01282,0.02499,0.04141,0.02652,0.04567,0.03295,0.03799,0.03613,0.02390,0.02358,0.03045,0.02376,0.01368,0.01010,0.01709,0.03519,0.01871,0.02561,0.03907,0.02111,0.03489,0.02021,0.03170,0.06113,0.04336,0.02258,0.03414,0.03310,0.04067,0.02289,0.02425,0.03052,0.03010,0.01762,0.02262,0.04684,0.05316,0.06113,0.01468,0.02063,0.01621,0.03425,0.03502,0.04710,0.03249,0.02528,0.05143,0.02769,0.03396,0.02458,0.03711,0.05316,0.02320,0.05775,0.02597,0.01537,0.06113,0.00689,0.01572,0.04710,0.05775,0.01422,0.02147,0.03957,0.04659,0.01066,0.04433,0.03065,0.03932,0.02751,0.02595,0.00664,0.05923,0.02735,0.02779,0.03983,0.02389,0.04057,0.03671,0.02961,0.02164,0.03167,0.01917,0.04336,0.05923


Sorted by tf-idf:


token,weapons,destruction,foxvog,mass,stratus,vtt,cdt,rutledge,fi,ulowell,of,the,sw,vttoulu,dfo,tko,douglas,term,needless,tavares,com,to,lawrence,says,transfer,individuals,doug,modern,1r1eu1,believe,you,f8f,4j3,1qv87v,c5n3gi,crimial,rigidly,16899,083057,article,right,cbw,ousrvr,she,her,sks,and,rewording,sweeper,in,existant,nukes,be,jrutledg,then,means,makes,keep,shotguns,john,allegedly,sarah,oulu,destructive,evidently,brady,4t,when,accidental,presenting,biological,that,writes,nerve,rifles,vos,for,checks,investors,blank,stating,switching,topics,usage,handguns,commonly,this,any,massive,packet,on,neighbor,restrictions,reduced,by,deaths,us,understood,many,first,semi,nuclear,millions,automatic,property,rocket,can,amendment,should,defined,disagree,analysis,thousands,sign,bear,immediately,58,follows,count,keeping,quote,hands,people,understanding,putting,street,1993apr20,gas,allowed,individual,class,reasonable,killed,speak,easily,ideas,doubt,argument,special,result,coming,uses,write,company,cost,later,consider,bill,show,agree,it,cannot,would,control,hope,my,given,was,today,an,support,non,access,government,second,power,mean,hard,as,each,number,great,every,course,year,or,if,read,must,own,point,another,using,find,back,cs,from,need,say,thanks,edu,his,where,re,even,these,he,them,does,other,we,only,up,don,out,has,about,all,what,there,are,not,with,have,organization,lines,subject
count,13,7,5,7,6,4,4,3,4,3,17,17,3,2,2,2,3,3,2,2,6,10,2,3,2,2,2,2,1,3,6,1,1,1,1,1,1,1,1,5,3,1,1,2,2,1,7,1,1,7,1,1,5,1,3,2,2,2,1,2,1,1,1,1,1,1,1,3,1,1,1,5,4,1,1,1,5,1,1,1,1,1,1,1,1,1,4,3,1,1,4,1,1,1,3,1,2,1,2,2,1,1,1,1,1,1,3,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,3,1,2,1,1,2,1,2,1,2,1,1,1,1,1,1,1,1,2,1,1,1,1,1,1,2,2,1,1,1,1,1,1,1,1,1,3,1,1,1,2,1,1,2,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
tfidf,0.42512,0.29575,0.28272,0.23218,0.22956,0.22209,0.16189,0.15281,0.13497,0.13298,0.12769,0.12030,0.11416,0.11309,0.11309,0.11309,0.10584,0.09510,0.08865,0.08328,0.07385,0.07338,0.07227,0.07089,0.06836,0.06784,0.06741,0.06546,0.06382,0.06343,0.06139,0.06113,0.06113,0.06113,0.06113,0.06113,0.06113,0.05923,0.05923,0.05837,0.05809,0.05775,0.05775,0.05681,0.05635,0.05552,0.05490,0.05464,0.05464,0.05431,0.05316,0.05316,0.05168,0.05143,0.05141,0.05035,0.05023,0.04828,0.04825,0.04779,0.04765,0.04737,0.04710,0.04710,0.04684,0.04659,0.04659,0.04568,0.04567,0.04567,0.04546,0.04453,0.04438,0.04433,0.04415,0.04382,0.04347,0.04336,0.04336,0.04225,0.04188,0.04176,0.04141,0.04087,0.04067,0.04057,0.03994,0.03985,0.03983,0.03957,0.03950,0.03932,0.03932,0.03915,0.03909,0.03907,0.03838,0.03799,0.03779,0.03754,0.03754,0.03711,0.03671,0.03660,0.03613,0.03589,0.03557,0.03551,0.03530,0.03519,0.03502,0.03498,0.03498,0.03498,0.03489,0.03489,0.03477,0.03425,0.03414,0.03396,0.03343,0.03310,0.03298,0.03295,0.03292,0.03279,0.03252,0.03249,0.03188,0.03170,0.03167,0.03122,0.03065,0.03063,0.03052,0.03045,0.03010,0.02988,0.02982,0.02974,0.02961,0.02871,0.02786,0.02779,0.02769,0.02751,0.02735,0.02712,0.02660,0.02652,0.02642,0.02597,0.02597,0.02595,0.02561,0.02555,0.02528,0.02511,0.02499,0.02495,0.02466,0.02458,0.02443,0.02425,0.02415,0.02390,0.02389,0.02376,0.02367,0.02358,0.02320,0.02289,0.02262,0.02258,0.02212,0.02204,0.02199,0.02169,0.02164,0.02147,0.02111,0.02110,0.02109,0.02063,0.02042,0.02021,0.01986,0.01917,0.01913,0.01904,0.01887,0.01871,0.01866,0.01817,0.01762,0.01756,0.01709,0.01644,0.01621,0.01572,0.01546,0.01537,0.01533,0.01468,0.01422,0.01368,0.01314,0.01282,0.01273,0.01269,0.01088,0.01066,0.01043,0.01010,0.00689,0.00664,0.00662


# Classifiers Performance Analysis

Instead of manually transforming and training the models separately, we can define a pipeline that does it all in one go.

## Comparing Classifiers
### We will compare 4 different classifiers on the data:
- Logistic Regression
- Decision Tree
- kNN
- SVM

### Let's see an example of a document from our data

In [ ]:
print('Sample document:')
print(X_train[0])